# First Diagnostic: `count_common` Changes Along a Route

This notebook checks whether, for the same `month`, `DayOfWeek`, and `HourSourceTime`, the value of `count_common` changes across stations on the same `route_id`.

A change in adjacent stations can indicate potential station skipping behavior.

In [1]:
import pandas as pd
from pathlib import Path

# Load the merged route-level dataset
csv_path = Path('..') / 'govData' / 'ride_data_merged.csv'
df = pd.read_csv(csv_path)

# Keep only columns needed for this first diagnostic
cols = [
    'month', 'route_id', 'DayOfWeek', 'HourSourceTime',
    'StopSequence_Rishui', 'StopCode', 'count_common'
]
work = df[cols].copy()

# Ensure numeric columns are correctly typed
for c in ['month', 'DayOfWeek', 'HourSourceTime', 'StopSequence_Rishui', 'count_common']:
    work[c] = pd.to_numeric(work[c], errors='coerce')

# Drop rows missing key fields
work = work.dropna(subset=['month', 'route_id', 'DayOfWeek', 'HourSourceTime', 'StopSequence_Rishui', 'count_common'])

print(f'Loaded rows for analysis: {len(work):,}')
work.head()

Loaded rows for analysis: 225,802


,month,route_id,DayOfWeek,HourSourceTime,StopSequence_Rishui,StopCode,count_common
0,3,37936,1,15,2,3301,39
1,3,37936,1,15,3,5987,39
2,3,37936,1,15,4,5988,39
3,3,37936,1,15,5,9921,39
4,3,37936,1,15,6,2540,39


In [ ]:
# Group definition requested: same month + day of week + hour on the same route
key = ['route_id', 'month', 'DayOfWeek', 'HourSourceTime']

# 1) Which groups have inconsistent count_common across stations?
group_check = (
    work.groupby(key)
    .agg(
        stations_in_group=('StopCode', 'nunique'),
        unique_count_common=('count_common', 'nunique')
    )
    .reset_index()
)

inconsistent_groups = group_check[group_check['unique_count_common'] > 1].copy()

print(f'Groups checked: {len(group_check):,}')
print(f'Groups with count_common changes across stations: {len(inconsistent_groups):,}')
display(inconsistent_groups.sort_values('unique_count_common', ascending=False).head(20))

# 2) Detect adjacent-station transitions where count_common changes
ordered = work.sort_values(key + ['StopSequence_Rishui'])
ordered['prev_count_common'] = ordered.groupby(key)['count_common'].shift(1)
ordered['prev_stop_code'] = ordered.groupby(key)['StopCode'].shift(1)
ordered['prev_stop_seq'] = ordered.groupby(key)['StopSequence_Rishui'].shift(1)

changes = ordered[
    ordered['prev_count_common'].notna() &
    (ordered['count_common'] != ordered['prev_count_common'])
].copy()

changes['delta_count_common'] = changes['count_common'] - changes['prev_count_common']

# Filter to stronger changes only
changes = changes[changes['delta_count_common'].abs() >= 3].copy()

result_cols = [
    'route_id', 'month', 'DayOfWeek', 'HourSourceTime',
    'prev_stop_seq', 'StopSequence_Rishui',
    'prev_stop_code', 'StopCode',
    'prev_count_common', 'count_common', 'delta_count_common'
]

print(f'Adjacent station transitions with abs(delta_count_common) >= 3: {len(changes):,}')
display(changes[result_cols].head(50))

# Optional compact summary: how often each route shows these transitions
route_summary = (
    changes.groupby('route_id')
    .size()
    .rename('num_changed_transitions')
    .sort_values(ascending=False)
    .reset_index()
)

display(route_summary.head(20))

Groups checked: 4,315
Groups with count_common changes across stations: 169


,route_id,month,DayOfWeek,HourSourceTime,stations_in_group,unique_count_common
1866,10802,1,2,24,42,3
1867,10802,1,2,25,42,3
179,5499,1,1,13,55,2
177,5499,1,1,11,55,2
182,5499,1,1,16,55,2
187,5499,1,2,5,55,2
188,5499,1,2,6,55,2
181,5499,1,1,15,55,2
173,5499,1,1,7,55,2
191,5499,1,2,9,55,2


Adjacent station transitions with count_common changes: 13,200


,route_id,month,DayOfWeek,HourSourceTime,prev_stop_seq,StopSequence_Rishui,prev_stop_code,StopCode,prev_count_common,count_common,delta_count_common
128001,5499,1,1,7,1.0,1,3324.0,3324,11.0,12,1.0
129829,5499,1,1,7,1.0,1,3324.0,3324,12.0,11,-1.0
128002,5499,1,1,7,2.0,2,6268.0,6268,11.0,12,1.0
129830,5499,1,1,7,2.0,2,6268.0,6268,12.0,11,-1.0
128003,5499,1,1,7,3.0,3,9907.0,9907,11.0,12,1.0
129831,5499,1,1,7,3.0,3,9907.0,9907,12.0,11,-1.0
128004,5499,1,1,7,4.0,4,9908.0,9908,11.0,12,1.0
129832,5499,1,1,7,4.0,4,9908.0,9908,12.0,11,-1.0
128005,5499,1,1,7,5.0,5,3097.0,3097,11.0,12,1.0
129833,5499,1,1,7,5.0,5,3097.0,3097,12.0,11,-1.0


,route_id,num_changed_transitions
0,10802,7265
1,5499,3402
2,10398,2533
